# 09 Error Analysis

Analyze validation errors for the final TF-IDF Logistic Regression model with A/B swap augmentation and swap-test averaging.

## 1. Imports and Path Setup

Set up libraries, paths, and output folders. This notebook only analyzes saved predictions and does not train any model.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)
sns.set_theme(style='whitegrid')

LABELS = [0, 1, 2]
LABEL_NAMES = ['A_win', 'B_win', 'tie']
LABEL_NAME_MAP = {0: 'A_win', 1: 'B_win', 2: 'tie'}

current_dir = Path.cwd().resolve()
if current_dir.name.lower() == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

oof_dir = project_root / 'outputs' / 'oof'
figures_dir = project_root / 'outputs' / 'figures'
logs_dir = project_root / 'outputs' / 'logs'
report_dir = project_root / 'report'

for output_dir in [oof_dir, figures_dir, logs_dir, report_dir]:
    output_dir.mkdir(parents=True, exist_ok=True)

prediction_path = oof_dir / 'tfidf_lr_ab_swap_valid_predictions.csv'
confusion_matrix_path = figures_dir / 'final_model_error_confusion_matrix.png'
tie_cases_path = oof_dir / 'error_tie_cases.csv'
length_analysis_path = logs_dir / 'error_length_analysis.csv'
high_confidence_error_path = oof_dir / 'high_confidence_error_cases.csv'
typical_error_cases_path = oof_dir / 'typical_error_cases_for_report.csv'
report_path = report_dir / 'error_analysis.md'

print(f'Project root: {project_root}')
print(f'Prediction file exists: {prediction_path.exists()} -> {prediction_path}')

if not prediction_path.exists():
    raise FileNotFoundError('Please check the final model prediction file: outputs/oof/tfidf_lr_ab_swap_valid_predictions.csv')

## 2. Read Prediction File and Check Columns

Load validation predictions and verify the required columns.

In [ ]:
predictions = pd.read_csv(prediction_path, encoding='utf-8-sig')

required_columns = {
    'id',
    'label',
    'label_name',
    'pred_label',
    'pred_label_name',
    'prob_A_win',
    'prob_B_win',
    'prob_tie',
    'prompt_clean',
    'response_a_clean',
    'response_b_clean',
    'response_a_char_len',
    'response_b_char_len',
    'response_len_diff',
}

missing_columns = sorted(required_columns - set(predictions.columns))
if missing_columns:
    print(f'Missing columns: {missing_columns}')
    raise ValueError(f'Prediction file missing columns: {missing_columns}')

predictions['label'] = predictions['label'].astype(int)
predictions['pred_label'] = predictions['pred_label'].astype(int)

print(f'predictions shape: {predictions.shape}')
print('predictions columns:')
print(predictions.columns.tolist())
display(predictions.head(3))

## 3. Basic Error Statistics

Add correctness and confidence fields, then summarize overall error behavior.

In [ ]:
probability_columns = ['prob_A_win', 'prob_B_win', 'prob_tie']

predictions['is_correct'] = predictions['label'] == predictions['pred_label']
predictions['pred_confidence'] = predictions[probability_columns].max(axis=1)
probability_matrix = predictions[probability_columns].to_numpy()
predictions['true_label_prob'] = probability_matrix[np.arange(len(predictions)), predictions['label'].to_numpy()]

total_count = len(predictions)
correct_count = int(predictions['is_correct'].sum())
error_count = total_count - correct_count
accuracy = correct_count / total_count
mean_confidence = predictions['pred_confidence'].mean()
correct_mean_confidence = predictions.loc[predictions['is_correct'], 'pred_confidence'].mean()
error_mean_confidence = predictions.loc[~predictions['is_correct'], 'pred_confidence'].mean()

basic_stats = pd.Series({
    'total_count': total_count,
    'correct_count': correct_count,
    'error_count': error_count,
    'accuracy': accuracy,
    'mean_pred_confidence': mean_confidence,
    'correct_mean_pred_confidence': correct_mean_confidence,
    'error_mean_pred_confidence': error_mean_confidence,
})

display(basic_stats)

## 4. Class-Level Error Analysis

Compare true and predicted class distributions, calculate per-class accuracy, and save the confusion matrix figure.

In [ ]:
true_label_distribution = predictions['label_name'].value_counts().reindex(LABEL_NAMES)
pred_label_distribution = predictions['pred_label_name'].value_counts().reindex(LABEL_NAMES)

per_class_accuracy = (
    predictions.groupby('label_name')['is_correct']
    .agg(sample_count='size', correct_count='sum', accuracy='mean')
    .reindex(LABEL_NAMES)
)

print('True label distribution:')
display(true_label_distribution)

print('Predicted label distribution:')
display(pred_label_distribution)

print('Per-class accuracy:')
display(per_class_accuracy)

cm = confusion_matrix(predictions['label'], predictions['pred_label'], labels=LABELS)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=LABEL_NAMES,
    yticklabels=LABEL_NAMES,
)
plt.title('Final Model Error Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig(confusion_matrix_path, dpi=150)
plt.show()

print(f'Saved confusion matrix: {confusion_matrix_path}')

## 5. Tie Class Analysis

Inspect errors involving the tie class from both true-label and predicted-label perspectives.

In [ ]:
true_tie_cases = predictions[predictions['label_name'] == 'tie']
pred_tie_cases = predictions[predictions['pred_label_name'] == 'tie']

true_tie_prediction_counts = true_tie_cases['pred_label_name'].value_counts().reindex(LABEL_NAMES, fill_value=0)
pred_tie_true_counts = pred_tie_cases['label_name'].value_counts().reindex(LABEL_NAMES, fill_value=0)

print('For true tie samples, predicted label counts:')
display(true_tie_prediction_counts)

print('For predicted tie samples, true label counts:')
display(pred_tie_true_counts)

tie_error_cases = predictions[
    ((predictions['label_name'] == 'tie') | (predictions['pred_label_name'] == 'tie'))
    & (~predictions['is_correct'])
].copy()
tie_error_cases = tie_error_cases.sort_values(['label_name', 'pred_label_name', 'pred_confidence'], ascending=[True, True, False])
tie_error_cases.to_csv(tie_cases_path, index=False, encoding='utf-8-sig')

print(f'Saved tie error cases: {tie_cases_path}')
print(f'tie_error_cases shape: {tie_error_cases.shape}')
display(tie_error_cases.head())

## 6. Response Length Difference Analysis

Analyze whether response length difference is associated with prediction correctness.

In [ ]:
predictions['abs_response_len_diff'] = predictions['response_len_diff'].abs()

correct_error_length_summary = predictions.groupby('is_correct').agg(
    sample_count=('id', 'size'),
    response_len_diff_mean=('response_len_diff', 'mean'),
    response_len_diff_median=('response_len_diff', 'median'),
    abs_response_len_diff_mean=('abs_response_len_diff', 'mean'),
    abs_response_len_diff_median=('abs_response_len_diff', 'median'),
).reset_index()

bins = [-1, 200, 500, 1000, np.inf]
labels = ['0-200', '200-500', '500-1000', '1000+']
predictions['abs_response_len_diff_bin'] = pd.cut(
    predictions['abs_response_len_diff'],
    bins=bins,
    labels=labels,
)

length_bin_summary = predictions.groupby('abs_response_len_diff_bin', observed=False).agg(
    sample_count=('id', 'size'),
    accuracy=('is_correct', 'mean'),
    mean_pred_confidence=('pred_confidence', 'mean'),
    mean_true_label_prob=('true_label_prob', 'mean'),
).reset_index()

length_analysis = pd.concat(
    [
        correct_error_length_summary.assign(analysis_type='correct_vs_error'),
        length_bin_summary.assign(analysis_type='abs_len_diff_bin'),
    ],
    ignore_index=True,
    sort=False,
)

length_analysis.to_csv(length_analysis_path, index=False, encoding='utf-8-sig')

print('Correct vs error length summary:')
display(correct_error_length_summary)

print('Length bin summary:')
display(length_bin_summary)

print(f'Saved length analysis: {length_analysis_path}')

## 7. High Confidence Error Cases

Find the highest-confidence wrong predictions and the samples with the lowest true-label probability.

In [ ]:
def add_short_text_columns(df):
    result = df.copy()
    result['prompt_short'] = result['prompt_clean'].fillna('').astype(str).str[:300]
    result['response_a_short'] = result['response_a_clean'].fillna('').astype(str).str[:500]
    result['response_b_short'] = result['response_b_clean'].fillna('').astype(str).str[:500]
    return result


error_cases = predictions[~predictions['is_correct']].copy()
high_confidence_wrong = error_cases.sort_values('pred_confidence', ascending=False).head(20)
lowest_true_label_prob = error_cases.sort_values('true_label_prob', ascending=True).head(20)

high_confidence_combined = pd.concat(
    [
        high_confidence_wrong.assign(error_rank_type='highest_pred_confidence_wrong'),
        lowest_true_label_prob.assign(error_rank_type='lowest_true_label_prob_wrong'),
    ],
    ignore_index=True,
).drop_duplicates(subset=['id', 'label', 'pred_label'])

high_confidence_combined = add_short_text_columns(high_confidence_combined)

error_display_columns = [
    'id',
    'label_name',
    'pred_label_name',
    'pred_confidence',
    'true_label_prob',
    'response_len_diff',
    'prompt_clean',
    'response_a_clean',
    'response_b_clean',
    'prompt_short',
    'response_a_short',
    'response_b_short',
    'error_rank_type',
]

high_confidence_combined[error_display_columns].to_csv(
    high_confidence_error_path,
    index=False,
    encoding='utf-8-sig',
)

print(f'Saved high confidence error cases: {high_confidence_error_path}')
print('Top high-confidence wrong cases:')
display(add_short_text_columns(high_confidence_wrong)[error_display_columns[:-1]].head(20))

print('Lowest true-label probability wrong cases:')
display(add_short_text_columns(lowest_true_label_prob)[error_display_columns[:-1]].head(20))

## 8. Typical Error Cases for Report

Sample representative error cases for the course report.

In [ ]:
typical_case_groups = []

case_specs = [
    ('true_tie_pred_A_win', (predictions['label_name'] == 'tie') & (predictions['pred_label_name'] == 'A_win')),
    ('true_tie_pred_B_win', (predictions['label_name'] == 'tie') & (predictions['pred_label_name'] == 'B_win')),
    ('true_A_win_pred_tie', (predictions['label_name'] == 'A_win') & (predictions['pred_label_name'] == 'tie')),
    ('true_B_win_pred_tie', (predictions['label_name'] == 'B_win') & (predictions['pred_label_name'] == 'tie')),
]

for case_type, mask in case_specs:
    group = predictions[mask].copy()
    group = group.sort_values('pred_confidence', ascending=False).head(5)
    if len(group) > 0:
        group['case_type'] = case_type
        typical_case_groups.append(group)

high_confidence_report_cases = high_confidence_wrong.copy().head(5)
high_confidence_report_cases['case_type'] = 'high_confidence_wrong_cases'
typical_case_groups.append(high_confidence_report_cases)

typical_error_cases = pd.concat(typical_case_groups, ignore_index=True)
typical_error_cases = add_short_text_columns(typical_error_cases)

typical_case_columns = [
    'case_type',
    'id',
    'label_name',
    'pred_label_name',
    'pred_confidence',
    'true_label_prob',
    'response_len_diff',
    'prompt_short',
    'response_a_short',
    'response_b_short',
]

typical_error_cases[typical_case_columns].to_csv(
    typical_error_cases_path,
    index=False,
    encoding='utf-8-sig',
)

print(f'Saved typical error cases: {typical_error_cases_path}')
display(typical_error_cases[typical_case_columns].head(25))

## 9. Write Markdown Error Analysis Report

Generate a concise Markdown report section for the course project.

In [ ]:
tie_accuracy = per_class_accuracy.loc['tie', 'accuracy']
a_accuracy = per_class_accuracy.loc['A_win', 'accuracy']
b_accuracy = per_class_accuracy.loc['B_win', 'accuracy']
worst_length_bin = length_bin_summary.sort_values('accuracy', ascending=True).iloc[0]
best_length_bin = length_bin_summary.sort_values('accuracy', ascending=False).iloc[0]

report_text = f"""# Error Analysis

This section analyzes validation errors for the final TF-IDF + Logistic Regression model with A/B swap augmentation and swap-test averaging.

## Overall Error Situation

- Total validation samples: {total_count}
- Correct samples: {correct_count}
- Error samples: {error_count}
- Accuracy: {accuracy:.6f}
- Mean prediction confidence: {mean_confidence:.6f}
- Mean confidence on correct samples: {correct_mean_confidence:.6f}
- Mean confidence on wrong samples: {error_mean_confidence:.6f}

The model is better than random guessing, but the error count is still high. This is expected because the task requires comparing two long answers with nuanced quality differences.

## Class-Level Findings

- A_win accuracy: {a_accuracy:.6f}
- B_win accuracy: {b_accuracy:.6f}
- tie accuracy: {tie_accuracy:.6f}

The tie class is difficult because it often requires recognizing that two answers are similarly good or similarly flawed. TF-IDF features mainly capture surface word patterns, so they may not reliably identify subtle equivalence in answer quality.

For true tie samples, the predicted label distribution was:

```text
{true_tie_prediction_counts.to_string()}
```

For predicted tie samples, the true label distribution was:

```text
{pred_tie_true_counts.to_string()}
```

## Response Length Difference

The analysis also examines response_len_diff and abs_response_len_diff. The bin with the lowest accuracy was {worst_length_bin['abs_response_len_diff_bin']} with accuracy {worst_length_bin['accuracy']:.6f}. The bin with the highest accuracy was {best_length_bin['abs_response_len_diff_bin']} with accuracy {best_length_bin['accuracy']:.6f}.

Large differences in response length can influence the model because TF-IDF features may correlate verbosity with quality. However, longer responses are not always better, and shorter responses are not always worse. This means response_len_diff is useful but also potentially misleading.

## High-Confidence Errors

High-confidence wrong predictions show that the model can be confident for the wrong reasons. These errors indicate that the model has learned useful lexical patterns, but it still does not truly understand answer quality, factual correctness, or deep reasoning.

## Limitations of the TF-IDF Model

- It depends heavily on surface lexical features.
- It is weak at judging factual correctness.
- It is weak at judging deep logical consistency.
- It remains limited for long-text comparison and subjective preference tasks.

Overall, the final TF-IDF model is a strong lightweight baseline, but the error analysis shows why this competition remains challenging and why deeper semantic models may be useful when enough compute and data are available.
"""

report_path.write_text(report_text, encoding='utf-8')

print(f'Saved report: {report_path}')
print(report_text)

## 10. Final Checks

Check saved outputs and finish.

In [ ]:
print('Saved outputs:')
for path in [
    confusion_matrix_path,
    tie_cases_path,
    length_analysis_path,
    high_confidence_error_path,
    typical_error_cases_path,
    report_path,
]:
    print(f'{path.exists()} -> {path}')

print('Error analysis finished successfully.')